Cell 1: Google Drive Mount
python

from google.colab import drive
drive.mount('/content/drive')

Purpose: Mounts Google Drive to access data files stored in the cloud.
Why it's needed: The dataset file is too large to upload directly to Colab, so we use Google Drive as storage.
What happens: After running, you'll click an authorization link and get a mount point at /content/drive/ where your files appear.

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Cell 2: Data Loading and Initial Inspection

Purpose: Loads the dataset and shows what the data looks like.
What it does:

Imports essential Python libraries for data science

Reads the CSV file from Google Drive

Shows the first 5 rows to understand the data structure
Output revealed: The data has 94,380 rows and 54 columns with vehicle sensor data, timestamps, and driver classes.

In [5]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('/content/drive/MyDrive/data.csv')


df.head()

,Fuel_consumption,Accelerator_Pedal_value,Throttle_position_signal,Short_Term_Fuel_Trim_Bank1,Intake_air_pressure,Filtered_Accelerator_Pedal_value,Absolute_throttle_position,Engine_soacking_time,Inhibition_of_engine_fuel_cut_off,Engine_in_fuel_cut_off,...,Acceleration_speed_-_Longitudinal,Indication_of_brake_switch_ON/OFF,Master_cylinder_pressure,Calculated_road_gradient,Acceleration_speed_-_Lateral,Steering_wheel_speed,Steering_wheel_angle,Time(s),Class,PathOrder
0,268.8,0.0,5.2,0.0,33,0,13.3,3,0,0,...,-8.5,1,325.5,0.0,-8.8,0,-3.4,1,A,1
1,243.2,0.0,6.1,0.0,40,0,13.7,3,0,0,...,0.1,1,0.9,0.0,-0.2,0,-3.6,2,A,1
2,217.6,0.0,5.2,0.0,41,0,13.7,3,0,0,...,0.1,1,0.9,0.0,-0.2,0,-3.6,3,A,1
3,204.8,0.0,4.7,0.0,38,0,13.3,3,0,0,...,0.1,1,0.9,0.0,-0.2,0,-3.6,4,A,1
4,217.6,0.0,5.7,0.0,40,0,13.7,3,0,0,...,0.1,1,0.9,0.0,-0.2,0,-3.5,5,A,1


**Cell 3: Column Removal (Feature Selection)**

Purpose: Removes irrelevant or redundant columns from the dataset.
Why remove these:

PathOrder: Just a sequence number, not useful for analysis

Duplicate columns: Like Engine_coolant_temperature.1

Highly specific engine parameters not useful for driver identification

Constant or near-constant columns
Result: Reduces from 54 to 36 columns.

In [6]:
import pandas as pd
drop_cols = [
    'PathOrder',
    'Minimum_indicated_engine_torque',
    'Maximum_indicated_engine_torque',
    'Torque_scaling_factor(standardization)',
    'Standard_Torque_Ratio',
    'Requested_spark_retard_angle_from_TCU',
    'TCU_requests_engine_torque_limit_(ETL)',
    'TCU_requested_engine_RPM_increase',
    'Fuel_Pressure',
    'Inhibition_of_engine_fuel_cut_off',
    'Engine_in_fuel_cut_off',
    'Glow_plug_control_request',
    'Activation_of_Air_compressor',
    'Engine_Idel_Target_Speed',
    'Clutch_operation_acknowledge',
    'Converter_clutch',
    'Gear_Selection',
    'Engine_coolant_temperature.1'  # Dupe of original coolant
]

# Apply drop (safe: checks existence, no error if missing)
dfc = df.drop(columns=[col for col in drop_cols if col in df.columns])

# Quick sanity: Print shapes and var check for further prunes
print(f"Original shape: {df.shape}")
print(f"Cleaned shape: {dfc.shape}")


Original shape: (94380, 54)
Cleaned shape: (94380, 36)


In [7]:
print(len(dfc.columns))
n_col = len(dfc.columns)

36


**Cell 5: Time Window Compression Function**

Purpose: Compresses multiple time-series rows into a single summary row.
How it works: Takes all rows in a time window (e.g., 10 seconds of data) and calculates the average for each sensor.
Example: If throttle readings are [5.2, 6.1, 5.2, 4.7] over 4 seconds, returns [5.3] as the average.
Why it's useful: Reduces data volume while preserving overall patterns.

In [8]:
import numpy as np
import math
def compress(matrix):
    np_matrix = np.array(matrix)
    ls = np_matrix.mean(axis=0).tolist()
    #return [ (1 - math.exp(-abs(v)) )**2 for v in ls]
    return  ls

cell 6
Purpose: The core data preprocessing function that cleans and organizes data.
What it does:

    Removes non-numeric columns (except 'Class')

    Removes low-variance columns using coefficient of variation threshold

    Groups data into time windows (e.g., every 10 seconds)

    Compresses each window using the compress() function

    Tracks which rows belong to which driver class
    Parameters:

    df_v: Input dataframe

    vr_th: Variance ratio threshold (0.1 = keep columns with CV > 10%)

    timing: Time window size in seconds



In [9]:
def clean(df_v, vr_th , timing = 5):
    cl_df = df_v.copy()
    cols_to_drop = []

    for col_n in cl_df.columns:
        col = cl_df[col_n]

        if col_n == "Class":
            continue

        if not pd.api.types.is_numeric_dtype(col):
            cols_to_drop.append(col_n)
            continue

        if col_n == "Time(s)":
            continue

        mean_val = col.mean()
        var_val = col.var()

        if pd.isna(var_val) or var_val == 0:
            cols_to_drop.append(col_n)
            continue

        std_val = np.sqrt(var_val)

        if abs(mean_val) > 1e-10:
            cv = std_val / abs(mean_val)

            if cv < vr_th:
                cols_to_drop.append(col_n)

    print(len(cols_to_drop))
    cl_df =  cl_df.drop(columns=cols_to_drop)
    matrix = []
    positions = {}
    ligne = []
    for index, row in cl_df.iterrows():
      t = row["Time(s)"]
      if t > 0 and t % timing == 0 :
        if row["Class"] in positions:
            positions[row["Class"]].append(len(matrix))
        else:
            positions[row["Class"]] = [len(matrix)]
        matrix.append(compress(ligne))
        ligne = []
      ligne.append([val for key, val in row.items() if key not in ['Time(s)', 'Class']])



    return cl_df , matrix ,  positions

In [10]:
dfv , matrix ,  positions = clean(dfc, 0.1 ,10)
print(len(matrix) , dfv.shape[0])


3
9420 94380


In [11]:
def drivermatrixes(matrix , positions):
  matrixes = []
  for driver in positions.keys():
    matrixes.append( [ [matrix[v] ]for v in positions[driver] ])
  return matrix



## Cell 9: Feature Engineering - Statistical Feature Extraction

**What this function does:** This is the most critical step in our analysis. It transforms raw vehicle sensor data into meaningful "driving signatures" that uniquely identify each driver.

**Think of it this way:** If raw sensor data is like watching a video of someone driving, this function extracts their "driving personality" from that video.

### Step 1: Making Data Comparable (Robust Scaling)
Instead of using traditional scaling that can be thrown off by extreme values (like a sudden hard brake), we use a more resilient method. We find the middle value (median) and the typical spread (IQR - middle 50% range) for each sensor, then adjust all values relative to these. This makes the data from different sensors and different drivers comparable on the same scale.

### Step 2: Weighting What Matters (Feature Importance)
Not all sensors are equally important for identifying drivers. A sensor that barely changes (like engine coolant temperature once warmed up) tells us less than one that varies a lot (like steering wheel angle). We automatically give more weight to sensors that show more variation across time, as these are likely more informative for distinguishing driving styles.

### Step 3: Extracting the "DNA" of Driving
For each 10-second driving window, we extract **6 key characteristics**:

**First 3 features (Current State):**
1. **Average Behavior** - What's the typical driving style in this window?
2. **Consistency** - How steady or variable is the driving?
3. **Asymmetry** - Does the driver tend to favor certain actions (like more acceleration than braking)?

**Next 3 features (Dynamic Changes):**
4. **Change Magnitude** - How much does driving behavior change from the previous window?
5. **Change Consistency** - Are changes smooth or abrupt?
6. **Change Direction** - Do changes tend to go more in one direction than another?

**Why these 6 features work:**  
The first three capture "what" the driver is doing (their static style), while the last three capture "how" they do it (their dynamic adjustments). Together, they form a complete fingerprint of driving behavior that's unique to each person, much like handwriting analysis.

**Output:** Each 10-second window of driving (with 36 different sensor readings) gets reduced to just 6 numbers. These 6 numbers contain the essence of the driving behavior in that window and are perfect for training machine learning models to recognize drivers.

In [75]:
import numpy as np
from scipy.stats import skew
def scale_vec(mat):
  mat = np.array(mat)
  n = len(mat[0])
  m = len(mat)
  scaled_mat = []
  variances = []
  weights = []
  def getrange(x):
    return [  mat[i][x] for i in range(m) ]
  for i in range(n):
    observations = getrange(i)
    median = np.median(observations)
    q1 = np.percentile(observations, 25)
    q3 = np.percentile(observations, 75)
    iqr = q3 - q1
    observations = (observations - median)/(0.001 +iqr)
    weights.append(1 / (np.var(observations) +  1e-10 ) )
    scaled_mat.append(list(observations))
  deltas = []
  sum_weights = sum(weights)
  weights= [ weights[i] /sum_weights  for i in range(n)]
  scaled = []
  for j in range(m - 1 ):
    observations = [ ( scaled_mat[i][j+1]-  scaled_mat[i][j]) * weights[i]  for i in range(n) ]
    obs = [ scaled_mat[i][j] * weights[i] for i in range(n)]
    scaled.append(obs)
    deltas.append(observations)

  #vectorize
  vecs = []
  def getInt(row):
    row = np.array(row)
    return np.mean(row), np.std(row) , skew(row)
  print(len(scaled_mat))
  for i in range(m-1):
    mx , stx , skx= getInt(scaled[i])
    my , sty , sky= getInt(deltas[i])
    vecs.append([ mx , stx , skx , my , sty , sky])
  return vecs






In [77]:
scale_vec(matrix)

31


[[np.float64(-0.015772770789745743),
  np.float64(0.02637446484089674),
  np.float64(0.8605398932574118),
  np.float64(0.0173561573281306),
  np.float64(0.03768606884642326),
  np.float64(1.9224678174368206)],
 [np.float64(0.001583386538384859),
  np.float64(0.04645964096891937),
  np.float64(2.550467910048838),
  np.float64(-0.004140119877876411),
  np.float64(0.02232208988570754),
  np.float64(-4.955219034633231)],
 [np.float64(-0.00255673333949155),
  np.float64(0.03154096060995469),
  np.float64(0.8441663308159855),
  np.float64(-0.005009523259051147),
  np.float64(0.024312372525420754),
  np.float64(-3.2589529955342567)],
 [np.float64(-0.0075662565985426975),
  np.float64(0.023340904354282627),
  np.float64(0.8601199501821525),
  np.float64(0.006248243841019576),
  np.float64(0.0150715179924284),
  np.float64(0.9781111482891616)],
 [np.float64(-0.0013180127575231214),
  np.float64(0.021461779809849372),
  np.float64(0.14962923920361154),
  np.float64(0.002899868202280746),
  np.fl